# **Emails Data Cleaning**

**Notebook:** `01_emails_data_cleaning.ipynb`  

---

## **Overview**

This notebook performs end-to-end cleaning and preparation of the **emails CRM dataset** (`emails.csv`).

## **Importing Packages and Modules**

In [6]:
import sys
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from pathlib import Path

## **Constants**

In [7]:
PATH_SEP = os.sep

## **Appending Paths**

In [8]:
# Merging the path
sys.path.append(os.path.abspath('../../'))

## **Loading Env**

In [9]:
# Load the .env file
load_dotenv()

True

## **Reading Data**

In [10]:
folder_path = os.getenv("RAW_DATA_FOLDER")
dir_files = os.listdir(folder_path)

# Creating File Paths
files = []
for file in dir_files:
    files.append(folder_path + PATH_SEP + file)

*Extracting the path from the location*

In [12]:
# Sorting Files for Ensuring Idempotency
files = sorted(files)

# Reading Data
emails = pd.read_csv(files[2])

emails.head()

C:\Users\ilakk\AppData\Local\Temp\ipykernel_25716\2015097123.py:5: DtypeWarning: Columns (0: crm_agent_chase_count, 1: crm_auto_renewal_status) have mixed types. Specify dtype option on import or set low_memory=False.
  emails = pd.read_csv(files[2])


,Co_Ref,Time_to_Renewal,crm_accreditation_completed,crm_timely_completion,crm_progress_towards_accreditation,crm_delays_in_accreditation,crm_contractor_suggested_leave,crm_contractor_engagement,crm_contractor_sentiment,crm_contractor_sentiment_score,...,crm_accreditation_issues,crm_membership_overdue,crm_auto_renewal_status,crm_dissatisified_with_renewal_price,crm_customer_complained,crm_refund_mentioned,crm_negative_customer_experience,crm_dissatisfaction_with_support,crm_financial_hardship_mentioned,year
0,KG5766,pre_renewal,Not Discussed,Not Discussed,Not Discussed,Yes,No,Yes,Neutral,50,...,Not Discussed,Yes,0,No,No,Yes,Yes,No,Yes,2025
1,EJ1532,14_out,Not Discussed,Not Discussed,Not Discussed,No,Not Discussed,No,Not Discussed,Not Discussed,...,Not Discussed,Not Discussed,0,Not Discussed,No,Yes,Yes,No,Not Discussed,2025
2,AA4063,prior_year,Not Discussed,Not Discussed,Not Discussed,No,No,Yes,Neutral,50,...,Not Discussed,No,0,No,No,Yes,Yes,Yes,Not Discussed,2025
3,JY9888,prior_year,No,No,Not Discussed,Yes,No,Yes,Satisfied,80,...,Not Discussed,Yes,0,Not Discussed,No,Yes,Yes,No,Not Discussed,2025
4,WO6689,pre_renewal,Not Discussed,Not Discussed,Not Discussed,No,No,Yes,Satisfied,80,...,No,No,0,No,No,Yes,Yes,No,Not Discussed,2026


## **EDA**

#### **Feature Documentation**

| Column Name | Data Type | Description |
|-------------|-----------|-------------|
| `Co_Ref` | object | Customer reference — primary key for aggregation |
| `Time_to_Renewal` | object | Stage in the renewal window (`pre_renewal`, `14_out`, `prior_year`, …) |
| `crm_accreditation_completed` | object | Whether accreditation was completed (`Yes`/`No`/`Not Discussed`) |
| `crm_timely_completion` | object | Whether accreditation was completed on time |
| `crm_progress_towards_accreditation` | object | Progress status towards accreditation |
| `crm_delays_in_accreditation` | object | Whether delays were reported — cleaned to `yes`/`no`/`not discussed` |
| `crm_contractor_suggested_leave` | object | Whether contractor suggested leaving |
| `crm_contractor_engagement` | object | Contractor engagement flag — cleaned to `yes`/`no`/`not discussed` |
| `crm_contractor_sentiment` | object | Free-text sentiment — normalised to `satisfied`/`dissatisfied`/`neutral`/`not discussed` |
| `crm_contractor_sentiment_score` | float64 | Numeric sentiment score (0–100); `Not Discussed` coerced to `NaN` then filled with `0` |
| `crm_dts_or_ssip_mentioned` | object | Whether DTS/SSIP was mentioned — rows outside `yes`/`no`/`not discussed` are dropped |
| `crm_customer_payment_intention` | object | Customer payment intent — rows outside allowed values are dropped |
| `crm_membership_overdue` | object | Whether membership is overdue — rows outside allowed values are dropped |
| `crm_membership_level` | object | Free-text membership tier — normalised to controlled categories |
| `crm_dissatisified_with_renewal_price` | object | Price dissatisfaction flag |
| `crm_customer_complained` | object | Complaint flag — cleaned to `yes`/`no`/`not discussed` |
| `crm_refund_mentioned` | object | Whether a refund was mentioned — cleaned to `yes`/`no`/`not discussed` |
| `crm_negative_customer_experience` | object | Negative experience flag — cleaned to `yes`/`no`/`not discussed` |
| `crm_dissatisfaction_with_support` | object | Support dissatisfaction flag — cleaned to `yes`/`no`/`not discussed` |
| `crm_financial_hardship_mentioned` | object | Whether financial hardship was mentioned — cleaned to `yes`/`no`/`not discussed` |
| `crm_competitors_mentioned` | object | Whether competitors were mentioned — cleaned to `yes`/`no`/`not discussed` |
| `crm_agent_chase_count` | float64 | Number of agent follow-up attempts; `Not Discussed` coerced to `NaN` then filled with `0` |
| `crm_auto_renewal_status` | object | Auto-renewal status flag |
| `year` | int64 | Renewal year — cast to numeric and averaged during aggregation |

---

## **Data Cleaning**

### **Checking and Casting Numeric Columns**

Two CRM columns arrive as mixed-type strings due to free-text LLM output:
- `crm_contractor_sentiment_score` — numeric score (0–100)
- `crm_agent_chase_count` — count of agent follow-up attempts

`pd.to_numeric(..., errors='coerce')` safely converts valid numbers and replaces unparseable strings with `NaN`.

In [13]:

numeric_cols = ['crm_contractor_sentiment_score', 'crm_agent_chase_count']
for col in numeric_cols:
    emails[col] = pd.to_numeric(emails[col], errors='coerce')

emails[numeric_cols].head()

,crm_contractor_sentiment_score,crm_agent_chase_count
0,50.0,1.0
1,NaN,1.0
2,50.0,0.0
3,80.0,3.0
4,80.0,0.0


### **Handling Null Values in Numeric Columns**

Missing values introduced by `coerce` above are filled with **0**.
A zero score signals the metric was not recorded rather than implying a negative value.

In [14]:
for col in numeric_cols:
    emails[col] = emails[col].fillna(0)

emails[numeric_cols].isnull().sum()

crm_contractor_sentiment_score    0
crm_agent_chase_count             0
dtype: int64

### **Normalising Categorical Columns**

All non-numeric columns (excluding `Co_Ref`) are standardised:
- **Cast to string** — eliminates residual mixed-type issues
- **Strip & lowercase** — removes whitespace and unifies casing
- **Replace null-like tokens** — `'nan'`, `''`, `'none'`, `np.nan` → `'Not Discussed'`

In [15]:

categorical_cols = emails.columns.difference(numeric_cols)

for col in categorical_cols[1:]:
    emails[col] = emails[col].astype(str).str.strip().str.lower()
    emails[col] = emails[col].replace(['nan', '', 'none', np.nan], 'Not Discussed')

emails[categorical_cols].head()

,Co_Ref,Time_to_Renewal,crm_accreditation_completed,crm_accreditation_issues,crm_agent_chased_contractor,crm_auto_renewal_status,crm_competitors_mentioned,crm_contractor_engagement,crm_contractor_sentiment,crm_contractor_suggested_leave,...,crm_dts_or_ssip_mentioned,crm_financial_hardship_mentioned,crm_membership_level,crm_membership_overdue,crm_negative_customer_experience,crm_platform_issues_raised,crm_progress_towards_accreditation,crm_refund_mentioned,crm_timely_completion,year
0,KG5766,pre_renewal,not discussed,not discussed,yes,0,no,yes,neutral,no,...,no,yes,not discussed,yes,yes,no,not discussed,yes,not discussed,2025
1,EJ1532,14_out,not discussed,not discussed,yes,0,no,no,not discussed,not discussed,...,no,not discussed,accredited,not discussed,yes,no,not discussed,yes,not discussed,2025
2,AA4063,prior_year,not discussed,not discussed,no,0,no,yes,neutral,no,...,no,not discussed,accredited,no,yes,no,not discussed,yes,not discussed,2025
3,JY9888,prior_year,no,not discussed,yes,0,no,yes,satisfied,no,...,yes,not discussed,accredited,yes,yes,yes,not discussed,yes,no,2025
4,WO6689,pre_renewal,not discussed,no,no,0,no,yes,satisfied,no,...,no,not discussed,accredited,no,yes,no,not discussed,yes,not discussed,2026


## **Handling Null Values**

### **Consolidated Cleaning — Yes / No / Not Discussed Columns**

Many CRM columns contain free-text LLM outputs that must be mapped to a three-category vocabulary:
**`yes`**, **`no`**, and **`not discussed`**.

**Approach:**
- `GLOBAL_RULES` captures keyword lists for each category
- `clean_text_to_category()` applies a **priority-ordered** scan: `not discussed` → `no` → `yes`
- Defaults to `'not discussed'` when no keyword matches

*Columns cleaned: `crm_delays_in_accreditation`, `crm_contractor_engagement`, `crm_customer_complained`,
`crm_financial_hardship_mentioned`, `crm_dissatisfaction_with_support`, `crm_negative_customer_experience`,
`crm_refund_mentioned`, `crm_competitors_mentioned`*

In [16]:
GLOBAL_RULES = {
    'yes': [
        'yes', 'negative', 'frustration', 'dissatisfied', 'unhappy',
        'issue', 'error', 'concern', 'complaint',
        'negative customer experience', 'awkward interaction', 'excessive information',
        'overcharging', 'delayed', 'unfulfilled promises', 'incorrect', 'misled',
        'price increase', 'lack of value', 'annoyance', 'struggle',
        'disproportionately affect', 'not met', 'not matching',
        'unwanted call', 'abrupt hang-up', 'shut it down', 'unfair', 'unsustainable',
        'concerns about the audit process', 'causing them to lose work',
        'due to the issue with', 'negative customer ,experience', 'confusion'
    ],

    'no': [
        'no', 'price is too high', 'cost', 'not feasible', 'cancel',
        'loss of interest', 'not the right time', 'dissatisfaction',
        'not wish to renew', 'drop clients', 'time constraints',
        'difficulties in responding', 'ridiculous',
        'unhappy with no-refund', 'threatening to look elsewhere',
        'change in business structure', 'irrelevant requests'
    ],

    'not discussed': [
        'not discussed',
        'just asked for invoice',
        'no email content provided',
        'there is no email content to analyze',
        'no conversation provided',
        '[yes/no/not discussed]',
        'not applicable'
    ],
}
def clean_text_to_category(text, rules):
    text_low = str(text).strip().lower()

    # Priority Order (VERY IMPORTANT)
    priority = ['not discussed','no', 'yes']

    for category in priority:
        for keyword in rules.get(category, []):
            if keyword in text_low:
                return category

    return 'not discussed'  # default
columns_to_clean = [
    'crm_delays_in_accreditation',
    'crm_contractor_engagement',
    'crm_customer_complained',
    'crm_financial_hardship_mentioned',
    'crm_dissatisfaction_with_support',
    'crm_negative_customer_experience',
    'crm_refund_mentioned',
    'crm_competitors_mentioned'
]

for col in columns_to_clean:
    emails[col] = emails[col].apply(lambda x: clean_text_to_category(x, GLOBAL_RULES))

    print(f"\nValue counts for '{col}':")
    print(emails[col].value_counts())


Value counts for 'crm_delays_in_accreditation':
crm_delays_in_accreditation
no               60816
yes              40318
not discussed    22255
Name: count, dtype: int64

Value counts for 'crm_contractor_engagement':
crm_contractor_engagement
yes              71130
no               31207
not discussed    21052
Name: count, dtype: int64

Value counts for 'crm_customer_complained':
crm_customer_complained
no               104316
not discussed     11505
yes                7568
Name: count, dtype: int64

Value counts for 'crm_financial_hardship_mentioned':
crm_financial_hardship_mentioned
not discussed    83723
no               33121
yes               6545
Name: count, dtype: int64

Value counts for 'crm_dissatisfaction_with_support':
crm_dissatisfaction_with_support
no               66347
not discussed    46308
yes              10734
Name: count, dtype: int64

Value counts for 'crm_negative_customer_experience':
crm_negative_customer_experience
no               66587
not discussed    38

### **Cleaning `crm_contractor_sentiment`**

Free-text sentiment is mapped to `satisfied` / `dissatisfied` / `neutral` / `not discussed`.
Negative keywords take precedence; standalone `yes`/`no` are treated as `satisfied`/`dissatisfied`.

In [ ]:
def clean_contractor_sentiment(text):
    text_low = str(text).strip().lower()
    dissatisfied_keywords = [
        'dissatisfied', 'cancellation', 'ridiculous', 'no work gained',
        'unhappy', 'price is too high', 'frustration', 'difficulties', 'cost',
        'not feasible to renew', 'not wish to renew', 'drop clients', 'time constraints',
        'threatening to look elsewhere', 'loss of interest', 'change in business structure',
        'irrelevant requests', 'not the right time', 'not happy', 'negative'
    ]
    for keyword in dissatisfied_keywords:
        if keyword in text_low:
            return 'dissatisfied'
    if text_low == 'yes':
        return 'satisfied'
    if text_low == 'no':
        return 'dissatisfied'
    if 'satisfied' in text_low or 'protecting people' in text_low or  'later satisfied' in text_low:
        return 'satisfied'
    if 'neutral' in text_low or 'later neutral' in text_low:
        return 'neutral'
    return 'not discussed'
emails['crm_contractor_sentiment'] = emails['crm_contractor_sentiment'].apply(clean_contractor_sentiment)

print(emails['crm_contractor_sentiment'].value_counts())

crm_contractor_sentiment
neutral          53576
not discussed    52202
satisfied        11426
dissatisfied      6185
Name: count, dtype: int64


### **Filtering Rows with Invalid Domain Values**

Rows where `crm_dts_or_ssip_mentioned`, `crm_customer_payment_intention`, or `crm_membership_overdue`
contain out-of-vocabulary values are removed to prevent noise in downstream features.

In [18]:
allowed_values = ['yes', 'no', 'not discussed']

initial_rows = emails.shape[0]

emails = emails[emails['crm_dts_or_ssip_mentioned'].isin(allowed_values)].copy()

emails = emails[emails['crm_customer_payment_intention'].isin(allowed_values)].copy()

emails = emails[emails['crm_membership_overdue'].isin(allowed_values)].copy()

### **Normalising `crm_membership_level`**

`clean_membership_level()` maps free-text membership tier descriptions to a controlled set:
`accredited`, `assisted`, `standard`, `silver`, `gold`, `bronze`, `premium`, `member`,
`in progress`, `express`, `band`, `not discussed`.

In [19]:
def clean_membership_level(text):
    text_low = str(text).strip().lower()

    if 'accredited' in text_low:
        return 'accredited'
    elif 'assist' in text_low:
        return 'assisted'
    elif 'standard' in text_low:
        return 'standard'
    elif 'silver' in text_low:
        return 'silver'
    elif 'gold' in text_low:
        return 'gold'
    elif 'bronze' in text_low:
        return 'bronze'
    elif 'prem' in text_low or 'premier' in text_low:
        return 'premium'
    elif 'member' in text_low or 'members' in text_low:
        return 'member'
    elif 'in progress' in text_low:
        return 'in progress'
    elif 'express' in text_low:
        return 'express'
    elif 'band' in text_low:
        return 'band'
    else:
        return 'not discussed'

emails['crm_membership_level']  = emails['crm_membership_level'].apply(clean_membership_level)

print("Value counts for 'crm_membership_level' after cleaning:")
print(emails['crm_membership_level'].value_counts())

Value counts for 'crm_membership_level' after cleaning:
crm_membership_level
in progress      36624
accredited       33420
not discussed    20529
member             542
standard            69
premium              8
gold                 6
express              4
silver               1
Name: count, dtype: int64


## **Aggregation — Removing `Co_Ref` Duplicates**

Multiple rows per customer are collapsed to one row per `Co_Ref`:

| Column Type | Aggregation Method |
|-------------|--------------------|
| Numeric (`crm_contractor_sentiment_score`, `crm_agent_chase_count`, `year`) | **Mean** |
| Categorical (all other columns) | **Mode** (defaults to `'not discussed'` if empty) |

In [20]:

emails['year'] = pd.to_numeric(emails['year'], errors='coerce')
numeric_cols = ['crm_contractor_sentiment_score', 'crm_agent_chase_count', 'year']
categorical_cols = [col for col in emails.columns if col not in numeric_cols and col != 'Co_Ref']
aggregation_dict = {}
for col in numeric_cols:
    aggregation_dict[col] = 'mean'
for col in categorical_cols:
    aggregation_dict[col] = lambda x: x.mode()[0] if not x.mode().empty else 'not discussed'
initial_rows = emails.shape[0]
emails_aggregated = emails.groupby('Co_Ref').agg(aggregation_dict).reset_index()
final_rows = emails_aggregated.shape[0]
emails = emails_aggregated
print(f"Initial rows: {initial_rows}")
print(f"Final rows after consolidating duplicates based on 'Co_Ref': {final_rows}")
print(f"Number of rows consolidated: {initial_rows - final_rows}")
display(emails.head())

Initial rows: 91203
Final rows after consolidating duplicates based on 'Co_Ref': 33745
Number of rows consolidated: 57458


,Co_Ref,crm_contractor_sentiment_score,crm_agent_chase_count,year,Time_to_Renewal,crm_accreditation_completed,crm_timely_completion,crm_progress_towards_accreditation,crm_delays_in_accreditation,crm_contractor_suggested_leave,...,crm_agent_chased_contractor,crm_accreditation_issues,crm_membership_overdue,crm_auto_renewal_status,crm_dissatisified_with_renewal_price,crm_customer_complained,crm_refund_mentioned,crm_negative_customer_experience,crm_dissatisfaction_with_support,crm_financial_hardship_mentioned
0,AA0584,50.000000,1.000000,2025.0,45_out,no,no,yes,no,no,...,no,no,not discussed,0,no,no,no,no,no,not discussed
1,AA0641,16.666667,0.666667,2025.0,14_out,no,not discussed,yes,yes,not discussed,...,no,not discussed,not discussed,0,not discussed,no,no,no,not discussed,not discussed
2,AA0750,50.000000,1.500000,2025.0,45_out,no,no,yes,no,no,...,yes,not discussed,no,0,not discussed,no,no,no,no,no
3,AA0794,16.666667,1.666667,2025.0,14_out,no,not discussed,yes,yes,no,...,yes,not discussed,yes,0,not discussed,no,no,no,no,no
4,AA0882,50.000000,2.000000,2025.0,45_out,no,no,yes,yes,no,...,yes,yes,not discussed,0,not discussed,no,no,no,no,not discussed


## **Validation — Duplicate Percentage for `Co_Ref` After Aggregation**

Verifies that the composite key `(Co_Ref, Time_to_Renewal, year)` is unique across all rows.
A duplicate percentage of **0.00%** confirms the aggregation was successful.

In [21]:
total_rows = emails.shape[0]

duplicate_rows = emails[emails.duplicated(subset=['Co_Ref','Time_to_Renewal','year'], keep=False)].shape[0]

percentage = (duplicate_rows / total_rows) * 100

print("Total rows:", total_rows)
print("Duplicated rows:", duplicate_rows)
print("Percentage of duplicates: {:.2f}%".format(percentage))

Total rows: 33745
Duplicated rows: 0
Percentage of duplicates: 0.00%


## **Testing Data Export**

The cleaned, deduplicated DataFrame is saved to `emails_cleaned.csv` and downloaded locally.
This file is the clean input for the feature engineering and modelling pipelines.

In [23]:

output_folder_path = os.getenv("CLEAN_DATA_FOLDER")
emails.to_csv(output_folder_path+PATH_SEP+"emails_clean.csv",index=False)